# 6주 1차시 · 평균·중앙값·분산

**학습 목표**
- 평균과 중앙값을 구하고, 극단값(이상치) 하나가 평균을 끌고 가는 모습을 확인한다.
- 범위·분산·표준편차로 데이터가 퍼진 정도를 숫자로 나타낸다.
- `describe()`로 요약 통계를 한 번에 구하고 읽는다.

**사용 데이터** — `process.csv` (가상 데이터, 494행)
사출 성형 공정 기록입니다. 제품마다 사출 온도(`temp`, ℃), 압력(`pressure`, bar), 습도(`humidity`, %), 두께(`thickness`, mm), 인장강도(`strength`, MPa), 불량 여부(`defect`, 0/1) 등이 기록되어 있습니다.
비교용으로 5주차에 정리하기 전 파일 `process_raw.csv`도 잠깐 씁니다.

## 0. 준비
한글 그래프용 패키지를 설치하고, pandas로 데이터를 불러옵니다. 셀을 차례로 실행(▶ 또는 Shift+Enter)하세요.

In [ ]:
!pip -q install koreanize-matplotlib
import koreanize_matplotlib
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
DATA = 'https://raw.githubusercontent.com/Daeil-Noh/Lectures/main/ax-microdegree/ai-data-analysis/data/'
df = pd.read_csv(DATA + 'process.csv')
print(df.shape)      # (행 수, 열 수)
df.head()

## 1. 평균과 중앙값
- **평균**(`mean`): 모두 더해서 개수로 나눈 값 — 시소의 균형점
- **중앙값**(`median`): 작은 값부터 줄 세웠을 때 한가운데 값

In [ ]:
s = df['strength']                    # 인장강도 열 하나 꺼내기
print('평균  :', round(s.mean(), 2))
print('중앙값:', round(s.median(), 2))

## 2. 값 하나가 고장 나면?
3호기 첫 제품 9개의 온도로 실험합니다. 마지막 값을 센서 오류값 **999.9**로 바꾸면 평균과 중앙값은 각각 얼마나 움직일까요?

In [ ]:
t9 = df[df['machine'] == '3호기']['temp'].head(9).reset_index(drop=True)
print('원래 값 :', t9.tolist())
print('평균', round(t9.mean(), 1), '/ 중앙값', round(t9.median(), 1))

t_err = t9.copy()
t_err[8] = 999.9                      # 9번째 값을 센서 오류값으로 바꿈
print('고장 후 :', t_err.tolist())
print('평균', round(t_err.mean(), 1), '/ 중앙값', round(t_err.median(), 1))

### 실제 데이터에서도
5주차에 정리하기 전 파일에는 온도 센서 오류값(999.9, −1.0)이 8개 있었습니다. 정리 전과 후를 비교해 봅니다.

In [ ]:
raw = pd.read_csv(DATA + 'process_raw.csv')
compare = pd.DataFrame({
    '정리 전(raw)': [raw['temp'].mean(), raw['temp'].median(), raw['temp'].std()],
    '정리 후':      [df['temp'].mean(),  df['temp'].median(),  df['temp'].std()],
}, index=['평균', '중앙값', '표준편차'])
compare.round(1)

오류값 몇 개 때문에 **평균과 표준편차**는 크게 달라졌지만 **중앙값**은 그대로입니다.
평균과 중앙값이 많이 다르면 → 극단값이나 오류값이 있는지 의심하고 원본을 들여다봅니다.

## 3. 퍼짐: 범위·분산·표준편차
- **범위** = 최댓값 − 최솟값
- **분산** = 편차(값 − 평균)를 제곱해서 평균한 값 (단위도 제곱이 됨)
- **표준편차** = √분산 → 원래 단위. “평균에서 보통 이만큼 떨어져 있다”

In [ ]:
print('범위    :', round(s.max() - s.min(), 2))
print('분산    :', round(s.var(), 2))
print('표준편차:', round(s.std(), 2))

9개 값으로 분산을 직접 계산해서 `var()`와 같은지 확인해 봅니다. pandas는 (개수 − 1)로 나눕니다.

In [ ]:
dev = t9 - t9.mean()                  # 편차: 각 값 − 평균
print('편차:', dev.round(1).tolist())
print('편차의 합:', round(dev.sum(), 1) + 0.0)    # 더하면 0 → 그래서 제곱한다
print('직접 계산한 분산:', round((dev ** 2).sum() / (len(t9) - 1), 2))
print('var() 결과     :', round(t9.var(), 2))
print('표준편차 std() :', round(t9.std(), 2))

## 4. 평균 ± 표준편차 안에 몇 개?
종 모양에 가까운 분포라면 평균 ± 1 표준편차 안에 대략 ⅔가 들어옵니다. 인장강도로 확인합니다.

In [ ]:
m, sd = s.mean(), s.std()
inside = s.between(m - sd, m + sd)    # 구간 안이면 True
print(f'{m - sd:.1f} ~ {m + sd:.1f} MPa 안: {inside.sum()}개 ({inside.mean() * 100:.1f}%)')

plt.figure(figsize=(8, 3))
plt.hist(s, bins=40, color='steelblue')
plt.axvspan(m - sd, m + sd, color='orange', alpha=0.2, label='평균 ± 1 표준편차')
plt.axvline(m, color='orange', label='평균')
plt.xlabel('인장강도 (MPa)'); plt.ylabel('제품 수'); plt.legend()
plt.show()

## 5. describe(): 요약 통계 한 번에
`count`(개수) · `mean`(평균) · `std`(표준편차) · `min`(최솟값) · `25%` · `50%`(중앙값) · `75%` · `max`(최댓값).
25%와 75%는 다음 차시(상자그림)에서 자세히 다룹니다.

In [ ]:
df.describe().round(2)

## 6. 직접 바꿔 보기
아래 셀의 값을 바꾸고 다시 실행해 보세요. 결과가 어떻게 달라지나요?

**(1)** `col`을 `'temp'`, `'humidity'`, `'thickness'`, `'pressure'`로 바꿔 보세요. 평균과 중앙값이 가장 많이 차이 나는 열은?

In [ ]:
col = 'strength'                      # ← 열 이름을 바꿔 보세요
x = df[col]
print(col, '평균', round(x.mean(), 3), '중앙값', round(x.median(), 3), '표준편차', round(x.std(), 3))

**(2)** `고장값`을 `250`, `500`, `5000`으로 바꿔 보세요. 평균과 중앙값은 각각 얼마나 움직이나요?

In [ ]:
고장값 = 999.9                         # ← 바꿔 보세요
t_test = t9.copy()
t_test[8] = 고장값
print('평균', round(t_test.mean(), 1), '/ 중앙값', round(t_test.median(), 1))

**(3)** `k`를 `2`, `3`으로 바꿔 보세요. 평균 ± k 표준편차 안에 몇 %가 들어오나요?

In [ ]:
k = 1                                 # ← 1, 2, 3
inside_k = s.between(m - k * sd, m + k * sd)
print(f'평균 ± {k} 표준편차 안: {inside_k.mean() * 100:.1f}%')

## 정리
- **평균**은 극단값·오류값에 끌려가고, **중앙값**은 잘 버틴다 → 둘을 함께 보고, 차이가 크면 원본을 확인한다.
- 가운데가 같아도 **퍼짐**은 다를 수 있다. 해석은 원래 단위인 **표준편차**로.
- `describe()` 한 줄로 개수·평균·표준편차·최솟값·사분위수·최댓값을 본다.

**생성형AI에게 물어볼 만한 질문**
- “pandas `describe()` 결과에서 평균은 220인데 중앙값은 211이야. 어떤 원인을 의심할 수 있고, 확인하는 코드를 알려 줘.”
- “pandas의 `std()`는 왜 (개수 − 1)로 나눠? 고등학생도 이해할 수 있게 예를 들어 설명해 줘.”

AI가 준 코드는 **직접 실행해 보고**, 숫자가 위 결과와 맞는지 확인하세요.